# NLP Lab 6 — Stage A1: Text Classification

**Activities 1–5 | Educational synthetic movie reviews dataset**

Run cells in order. The supplied CSV is **not an official dataset**; it is a self-contained teaching sample created because no dataset was provided. Metrics on it do not represent real-world performance. Place this notebook next to the CSV and TXT files.

## Setup

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline

DATA_DIR = Path.cwd()


## Activity 1 — Bag-of-Words

In [ ]:
df = pd.read_csv(DATA_DIR / "movie_reviews.csv")
print(df.head())

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df["text"])
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("First 5 documents × first 10 features:\n", X_bow[:5, :10].toarray())


## Activity 2 — TF-IDF and raw word counts

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])

# Select common words that actually occur in the chosen document.
from collections import Counter
common = [word for word, _ in Counter(" ".join(df["text"].str.lower()).split()).most_common() if word in vectorizer.vocabulary_]
row = 1
words = [word for word in common if X_bow[row, vectorizer.vocabulary_[word]] > 0][:2]
print("Document:", df["text"].iloc[row])
for word in words:
    count = X_bow[row, vectorizer.vocabulary_[word]]
    weight = X_tfidf[row, tfidf_vectorizer.vocabulary_[word]]
    print(f"{word}: count={count}, TF-IDF={weight:.4f}")
print("TF-IDF downweights words used across many documents; values are normalized, so raw counts and TF-IDF weights are on different scales.")


## Activity 3 — Train both classifiers

The split occurs **before TF-IDF fitting** to avoid leaking test-set statistics into training.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
train_vectorizer = TfidfVectorizer()
X_train = train_vectorizer.fit_transform(X_train_text)
X_test = train_vectorizer.transform(X_test_text)

nb_model = MultinomialNB().fit(X_train, y_train)
lr_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)
lr_preds = lr_model.predict(X_test)
print("Naive Bayes predictions:", nb_preds)
print("Logistic Regression predictions:", lr_preds)


## Activity 4 — Metrics and confusion matrices

In [ ]:
def evaluate_model(name, y_true, y_pred):
    scores = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label="positive", zero_division=0),
    }
    print(f"\n{name}")
    for metric, value in scores.items():
        print(f"{metric.title()}: {value:.4f}")
    print("Confusion matrix [negative, positive]:\n", confusion_matrix(y_true, y_pred, labels=["negative", "positive"]))
    return scores["f1"]

nb_f1 = evaluate_model("Naive Bayes", y_test, nb_preds)
lr_f1 = evaluate_model("Logistic Regression", y_test, lr_preds)


## Activity 5 — Pipeline and unseen review predictions

In [ ]:
best_model = LogisticRegression(max_iter=1000) if lr_f1 >= nb_f1 else MultinomialNB()
pipeline = Pipeline([("tfidf", TfidfVectorizer()), ("classifier", best_model)])
pipeline.fit(X_train_text, y_train)

with open(DATA_DIR / "new_reviews.txt", encoding="utf-8") as f:
    new_reviews = [line.strip() for line in f if line.strip()]

for review, prediction in zip(new_reviews, pipeline.predict(new_reviews)):
    print(f"Review: {review}\nPrediction: {prediction}\n")
